In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchmetrics.regression import R2Score

from rvsablation.utils import PrecomputedLatentDataset, seed_everything

from pathlib import Path

import os

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_TYPE = "jepa"
RUN_NAME = "fsq_v_jepa_run_1"
EPOCHS = 50
LR = 1e-3
BATCH = 2048
NUM_TUBELETS = 4

rng = seed_everything(96)

🌱 Global seed set to 96


# Linear Action Probe

We train a linear probe over the flattened learned representations to predict the $6$-dim action vector from the SAC agent.

In [2]:
def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for batch in dataloader:
        latents = batch['latents'].to(device)
        latents = latents.reshape(latents.shape[0], -1)
        targets = batch['actions'][:, -1].to(device)

        optimizer.zero_grad()
        preds = model(latents)
        loss = criterion(preds, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


def evaluate_epoch(model, dataloader, criterion, r2_metric, device):
    model.eval()
    total_loss = 0.0
    r2_metric.reset()
    
    with torch.no_grad():
        for batch in dataloader:
            latents = batch['latents'].to(device)
            latents = latents.reshape(latents.shape[0], -1)
            targets = batch['actions'][:, -1].to(device)
            
            preds = model(latents)
            loss = criterion(preds, targets)
            total_loss += loss.item()
            
            r2_metric.update(preds, targets)
            
    avg_loss = total_loss / len(dataloader)
    
    r2_per_dim = r2_metric.compute() 
    mean_r2 = r2_per_dim.mean().item()
    
    return avg_loss, mean_r2, r2_per_dim.cpu().numpy()

## Load Precomputed Latents

In [3]:
print(f"⚙️ Using latents from {RUN_NAME}")

main_dir = Path("..") / f"{MODEL_TYPE}_runs" / RUN_NAME

data_dir = main_dir / "latents"
data_dir = data_dir.resolve()

print(f"📁 Loading latents from {data_dir}")

train_dataset = PrecomputedLatentDataset(data_dir / "training")
test_dataset = PrecomputedLatentDataset(data_dir / "testing")

print(f"↪\tTrain set length: {len(train_dataset)}")
print(f"↪\tTest set length: {len(test_dataset)}")
train_loader = DataLoader(train_dataset, batch_size=BATCH, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH, shuffle=False)

⚙️ Using latents from fsq_v_jepa_run_1
📁 Loading latents from /home/ron/dev/croce_lab/reprod_vs_semantic/jepa_runs/fsq_v_jepa_run_1/latents
↪	Train set length: 34980
↪	Test set length: 15000


In [6]:
TASK_DIM = train_dataset[0]["actions"][-1].shape[0]
REP_N, REP_D = train_dataset[0]["latents"].shape
IN_DIM = REP_N * REP_D
print(f"Predicting state observations of {TASK_DIM} dimensions via linear probe over inputs of {IN_DIM} dimensions.")

Predicting state observations of 6 dimensions via linear probe over inputs of 1176 dimensions.


## Train & Evaluate Linear Action Probe

In [5]:
probe = nn.Linear(IN_DIM, TASK_DIM).to(DEVICE)
optimizer = optim.Adam(probe.parameters(), lr=LR)
criterion = nn.MSELoss()
metric = R2Score(multioutput="raw_values").to(DEVICE)

print(f"--- Training {RUN_NAME.upper()} Action Decoding Probe ---")
for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(probe, train_loader, optimizer, criterion, DEVICE)
    val_loss, val_r2, r2_breakdown = evaluate_epoch(probe, test_loader, criterion, metric, DEVICE)

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d} | Train MSE: {train_loss:.4f} | Val MSE: {val_loss:.4f} | Val Mean R2: {val_r2:.4f}")

print("\n--- Final Dimensional R2 Breakdown ---")
# This matches up with the 24 dimensions of the Walker state
for i, r2_val in enumerate(r2_breakdown):
    print(f"Dim {i:02d}: {r2_val:.4f}")
print(f"Mean R^2: {val_r2}")

os.makedirs(main_dir / "action_probe", exist_ok=True)
torch.save(probe.state_dict(), main_dir / "action_probe" / "linear_probe.pt")
print(f"\n\n💾 Saved probe to {main_dir / "action_probe" / "linear_probe.pt"}.")

--- Training FSQ_V_JEPA_RUN_1 Action Decoding Probe ---
Epoch 001 | Train MSE: 0.8455 | Val MSE: 0.6054 | Val Mean R2: -0.2028
Epoch 005 | Train MSE: 0.4401 | Val MSE: 0.4467 | Val Mean R2: 0.1092
Epoch 010 | Train MSE: 0.4025 | Val MSE: 0.4167 | Val Mean R2: 0.1693
Epoch 015 | Train MSE: 0.3848 | Val MSE: 0.4076 | Val Mean R2: 0.1865
Epoch 020 | Train MSE: 0.3759 | Val MSE: 0.3979 | Val Mean R2: 0.2067
Epoch 025 | Train MSE: 0.3675 | Val MSE: 0.3948 | Val Mean R2: 0.2137
Epoch 030 | Train MSE: 0.3641 | Val MSE: 0.3910 | Val Mean R2: 0.2191
Epoch 035 | Train MSE: 0.3647 | Val MSE: 0.3867 | Val Mean R2: 0.2291
Epoch 040 | Train MSE: 0.3580 | Val MSE: 0.3866 | Val Mean R2: 0.2285
Epoch 045 | Train MSE: 0.3592 | Val MSE: 0.3860 | Val Mean R2: 0.2302
Epoch 050 | Train MSE: 0.3574 | Val MSE: 0.3849 | Val Mean R2: 0.2316

--- Final Dimensional R2 Breakdown ---
Dim 00: 0.2325
Dim 01: 0.1974
Dim 02: 0.1360
Dim 03: 0.2109
Dim 04: 0.1972
Dim 05: 0.4155
Mean R^2: 0.23159294444937434


💾 Saved pro